In [1]:
import sys, json, platform, importlib.metadata
from pathlib import Path
assert sys.version.split()[0] == '3.13.14', sys.version
Path('C:\\Users\\felip\\Documents\\Polimi\\Semester 3\\SE2 Quantum Satisfiability Project\\satelite-optimization-quantum-annealing\\PoliMi-SWE2-research-project\\results\\runs\\mwis-dominance-enhanced\\execution_logs\\exp2_s500_g50\\mwis_kernel_environment.json').write_text(json.dumps({'python':sys.version,'executable':sys.executable,'platform':platform.platform(),'packages':{name:importlib.metadata.version(name) for name in ['numpy', 'pandas', 'networkx', 'scipy', 'dwave-samplers', 'dimod', 'nbclient', 'nbformat', 'ipykernel', 'jupyter-client']}}, indent=2), encoding='utf-8')


Out[0]: 512


# MWIS pipeline with exact dominance reduction

Controlled experimental variant of `02.A_mwis_pipeline.ipynb`. The conflict rules, separator algorithm, budgets, QUBO and classical terminal solvers are unchanged. Outputs go to `runs/mwis-dominance-enhanced`; existing runs remain the control.

## 1. Configuration

All run parameters (input file, run name, QPU limits, decomposition budgets, exact-solver limits, QUBO coefficients) are read from `results/_config.json`. Set `run_name_mwis` to select the MWIS experiment; the MWIS graphics notebook uses the same setting. A copy of the configuration used is saved as `run_config_log.json` in `results/runs/mwis/pipeline_results/<run_name_mwis>`.

The shared annealing notebook selects its input independently through `qubo_input_run`. After saving the MWIS results, this notebook reports whether that setting points to this experiment and shows the path to use if necessary. It does not change the shared configuration.

In [2]:
from __future__ import annotations

import gzip
import json
import math
import os
import time
from collections import Counter
from itertools import combinations
from pathlib import Path
from typing import Any

import networkx as nx
import pandas as pd

def find_project_root(start: Path | None = None) -> Path:
    """
    Find the repository root so paths work without further trouble.
    """
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists() and (candidate / "results").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing both `data/` and `results/`.")

# loading project root dir
PROJECT_ROOT = find_project_root()
RESULTS_DIR = PROJECT_ROOT / "results"


# --------- loading configuration environment ----------
# load .json config file
CONFIG_PATH = Path(os.environ.get("SWE2_CONFIG", RESULTS_DIR / "_config.json"))
run_config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))

# load datset path
CANDIDATE_SLOT_CSV = PROJECT_ROOT / run_config["candidate_slot_csv"]

# MWIS experiment name and output directory
RUN_NAME = run_config["run_name_mwis"]
RUN_DIR = RESULTS_DIR / "runs" / "mwis-dominance-enhanced" / "pipeline_results" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

# QPU physical constraints
MAX_QPU_TERMINAL_NODES = run_config["max_qpu_terminal_nodes"]
MAX_QPU_TERMINAL_COUPLERS = run_config["max_qpu_terminal_couplers"]

# Separator-search limits. 
# If these limits are exceeded, the unresolved part is solved heuristically and clearly reported as non-exact.
MAX_SEPARATOR_SIZE = run_config["max_separator_size"]
MAX_SEPARATOR_CANDIDATE_POOL = run_config["max_separator_candidate_pool"]
MAX_SEPARATOR_STATES = run_config["max_separator_states"]
MAX_RECURSION_DEPTH = run_config["max_recursion_depth"]
MAX_DECOMPOSITION_NODES = run_config["max_decomposition_nodes"]

# Exact classical terminal-solver guards.
EXACT_ENUMERATION_NODE_LIMIT = run_config["exact_enumeration_node_limit"]
EXACT_CLIQUE_NODE_LIMIT = run_config["exact_clique_node_limit"]

# QUBO coefficients: scale the reward (linear) and the conflict penalty (quadratic).
# The penalty is computed from the effective reward scale, so changing the reward factor
# cannot accidentally make invalid conflicting selections attractive.
REWARD_FACTOR = float(run_config["qubo_reward_factor"])
PENALTY_FACTOR = float(run_config["qubo_penalty_factor"])
if not math.isfinite(REWARD_FACTOR) or REWARD_FACTOR <= 0:
    raise ValueError("qubo_reward_factor must be a positive finite number.")
if not math.isfinite(PENALTY_FACTOR) or PENALTY_FACTOR < 1.0:
    raise ValueError("qubo_penalty_factor must be a finite number >= 1.0 to preserve the MWIS penalty guarantee.")

# Configure logging for this run.
(RUN_DIR / "run_config_log.json").write_text(json.dumps(run_config, indent=2), encoding="utf-8")

print(f"Loaded configuration from: {CONFIG_PATH.resolve()}")
print(f"MWIS experiment: {RUN_NAME}")
print(f"Input dataset: {CANDIDATE_SLOT_CSV.resolve()}")
print(f"Artifacts will be saved to: {RUN_DIR.resolve()}")

PHASE_PROFILE_PATH = RUN_DIR / f'mwis_phase_profile_{time.strftime("%Y%m%d_%H%M%S")}.jsonl'
PIPELINE_STARTED = time.perf_counter()

def record_phase_timing(phase: str, started: float, **details: Any) -> float:
    """
    Helper function.
    Get how much time the specified phase takes to run, and store it in the log file.
    """
    elapsed = time.perf_counter() - started
    event = {"phase": phase, "elapsed_seconds": round(elapsed, 3), **details}
    with PHASE_PROFILE_PATH.open("a", encoding="utf-8") as profile_file:
        profile_file.write(json.dumps(event) + "\n")
        profile_file.flush()
    print(f"[MWIS timing] {phase}: {elapsed:.2f}s", flush=True)
    return elapsed

Loaded configuration from: C:\Users\felip\Documents\Polimi\Semester 3\SE2 Quantum Satisfiability Project\satelite-optimization-quantum-annealing\PoliMi-SWE2-research-project\results\runs\mwis-dominance-enhanced\execution_logs\exp2_s500_g50\config.json
MWIS experiment: exp2_s500_g50
Input dataset: C:\Users\felip\Documents\Polimi\Semester 3\SE2 Quantum Satisfiability Project\satelite-optimization-quantum-annealing\PoliMi-SWE2-research-project\data\experiment_inputs\exp_snapshot_2026-10-12\slots_s500_g50.csv
Artifacts will be saved to: C:\Users\felip\Documents\Polimi\Semester 3\SE2 Quantum Satisfiability Project\satelite-optimization-quantum-annealing\PoliMi-SWE2-research-project\results\runs\mwis-dominance-enhanced\pipeline_results\exp2_s500_g50


## 2. Load candidate slots

The data-wrangling notebooks already created weighted candidate contacts. Here we keep the essential columns used by the MWIS model.

In [3]:
REQUIRED_COLUMNS = [
    "slot_id", "source_interval_id", "satellite_id", "antenna_id", "station_id",
    "start_time_hours", "end_time_hours", "duration_hours",
    "reconfiguration_time_hours", "priority", "urgency", "weight",
]

phase_started = time.perf_counter()
slots = pd.read_csv(CANDIDATE_SLOT_CSV)[REQUIRED_COLUMNS]
slots.to_csv(RUN_DIR / "candidate_slots.csv", index=False)
record_phase_timing("candidate_load_and_export", phase_started, candidate_slots=len(slots))

print({
    "candidate_slots": len(slots),
    "satellites": slots["satellite_id"].nunique(),
    "antennas": slots["antenna_id"].nunique(),
    "source_intervals": slots["source_interval_id"].nunique(),
})

[MWIS timing] candidate_load_and_export: 0.08s
{'candidate_slots': 5224, 'satellites': 500, 'antennas': 50, 'source_intervals': 5224}


## 3. Build the MWIS conflict graph

Each candidate contact becomes one node. An edge means two contacts cannot both be selected:

- **same antenna**: the two contacts overlap in time, or the gap between them is shorter than the antenna reconfiguration time;
- **same satellite**: every satellite is served at most once over the whole horizon, so all the contacts of one satellite are pairwise in conflict (a clique). This is the same rule as the CSP pipeline, where each satellite takes at most one slot.

Therefore, a feasible schedule is an independent set, and the best schedule is a Maximum Weight Independent Set.

In [4]:
# ----------------------------------------------------------
# Build the conflict graph.
# ----------------------------------------------------------

def intervals_overlap(a_start: float, a_end: float, b_start: float, b_end: float) -> bool:
    """
    Return whether two half-open time intervals overlap.
    """
    return a_start < b_end and b_start < a_end


def gap_between(a: pd.Series, b: pd.Series) -> float:
    """
    Return the idle time between two non-overlapping intervals.
    """
    return b["start_time_hours"] - a["end_time_hours"] \
        if a["end_time_hours"] <= b["start_time_hours"]  \
            else a["start_time_hours"] - b["end_time_hours"] 


def add_conflict_edge(graph: nx.Graph, u: str, v: str, reason: str, **attrs: Any) -> None:
    """
    Add a labelled conflict edge unless that pair is already connected.
    """
    if u == v or graph.has_edge(u, v):
        return
    graph.add_edge(u, v, reason=reason, **attrs)


def build_conflict_graph(slot_df: pd.DataFrame) -> nx.Graph:
    """Create one node per candidate and edges for every incompatible pair.

    Conflicts represent antenna overlap or setup time, and any two contacts of
    the same satellite (each satellite is served at most once over the horizon).
    """
    graph = nx.Graph()
    microseconds_per_hour = 3_600_000_000

    # Each node represents one candidate contact slot.
    for row in slot_df.itertuples(index=False):
        graph.add_node(
            row.slot_id,
            source_interval_id=row.source_interval_id,
            satellite_id=str(row.satellite_id),
            antenna_id=row.antenna_id,
            station_id=row.station_id,
            start_time=float(row.start_time_hours),
            end_time=float(row.end_time_hours),
            duration=float(row.duration_hours),
            reconfiguration_time=float(row.reconfiguration_time_hours),
            priority=float(row.priority),
            urgency=float(row.urgency),
            weight=float(row.weight),
        )

    # Same-antenna temporal conflicts.
    for _, group in slot_df.groupby("antenna_id", sort=False):
        antenna_slots = group.sort_values("start_time_hours").reset_index(drop=True)
        rows = [row for _, row in antenna_slots.iterrows()]
        # Check pairs of slots assigned to the same antenna.
        for i, first in enumerate(rows):
            for second in rows[i + 1:]:

                # Overlapping contacts cannot use the same antenna.
                if intervals_overlap(
                    first["start_time_hours"], first["end_time_hours"],
                    second["start_time_hours"], second["end_time_hours"],
                ):
                    add_conflict_edge(graph, first["slot_id"], second["slot_id"], "same-antenna overlap")

                # if between intervals there is not enough time for reconfiguration, then add conflict.
                # switch to microseconds for better gap-control.
                elif round(gap_between(first, second) * microseconds_per_hour) < round(
                    max(first["reconfiguration_time_hours"], second["reconfiguration_time_hours"])
                    * microseconds_per_hour
                ):
                    add_conflict_edge(
                        graph, first["slot_id"], second["slot_id"],
                        "same-antenna reconfiguration time",
                        reconfiguration_time=max(first["reconfiguration_time_hours"], second["reconfiguration_time_hours"]),
                    )

    # A satellite is served at most once over the whole horizon: all its contacts are
    # pairwise in conflict (this also covers slots cut from the same visibility window).
    for satellite_id, group in slot_df.groupby("satellite_id", sort=False):
        for u, v in combinations(group["slot_id"].tolist(), 2):
            add_conflict_edge(graph, u, v, "same satellite", satellite_id=str(satellite_id))

    return graph
# End conflict-graph construction.

phase_started = time.perf_counter()
conflict_graph = build_conflict_graph(slots)
record_phase_timing("conflict_graph_build_and_export", phase_started, nodes=conflict_graph.number_of_nodes(), edges=conflict_graph.number_of_edges())

node_rows = [{"slot_id": node_id, **data} for node_id, data in conflict_graph.nodes(data=True)]
edge_rows = [
    {"source": u, "target": v, **data}
    for u, v, data in conflict_graph.edges(data=True)
]
# Save the graph tables for inspection and downstream notebooks.
pd.DataFrame(node_rows).to_csv(RUN_DIR / "original_conflict_graph_nodes.csv", index=False)
pd.DataFrame(edge_rows).to_csv(RUN_DIR / "original_conflict_graph_edges.csv", index=False)

print({
    "nodes": conflict_graph.number_of_nodes(),
    "edges_couplers": conflict_graph.number_of_edges(),
    "components": nx.number_connected_components(conflict_graph),
    "edge_reasons": dict(Counter(data["reason"] for _, _, data in conflict_graph.edges(data=True))),
})

[MWIS timing] conflict_graph_build_and_export: 4.90s
{'nodes': 5224, 'edges_couplers': 53998, 'components': 4, 'edge_reasons': {'same-antenna overlap': 8448, 'same-antenna reconfiguration time': 7146, 'same satellite': 38404}}


## Exact dominance reduction and independent components

Before splitting the graph, remove a contact only if another contact of the same satellite has at least its weight and no additional conflicts with other satellites. Because contacts of each satellite form a clique, replacing the removed contact by its witness preserves feasibility and cannot decrease the objective. Equivalent contacts retain the smallest slot identifier. Repeat until no removal is possible, then split the reduced graph into connected components. This reduction preserves the optimum but can change component boundaries.

The full input graph is saved in `original_conflict_graph_*.csv`. The downstream `conflict_graph_*.csv` tables describe the reduced solver graph, so reconstruction and MILP use the reduced components consistently. Dominated contacts are removed alternatives and are never added back to the schedule. Removal witnesses are saved for independent validation.

In [5]:
def reduce_same_satellite_dominance(graph: nx.Graph) -> dict[str, Any]:
    """Remove dominated alternatives to a fixed point, preserving MWIS value.

    Same-satellite alternatives must form cliques. A contact a is removable
    when b has at least its weight and a superset of b's external conflicts.
    Equivalent alternatives retain the lexicographically smallest slot ID.
    Each pass uses a snapshot; a witness removed earlier in that pass is skipped.
    Removal records are local to this input graph, including recursive branches.
    """
    groups: dict[str, set[str]] = {}
    for v, data in graph.nodes(data=True):
        groups.setdefault(str(data["satellite_id"]), set()).add(v)
    for group in groups.values():
        if any(len(set(graph.neighbors(v)) & group) != len(group) - 1 for v in group):
            raise ValueError("Dominance requires same-satellite alternatives to form a clique.")
    live = set(graph.nodes)
    records = []
    passes = 0
    while True:
        external = {v: (set(graph.neighbors(v)) & live) - groups[str(graph.nodes[v]["satellite_id"])] for v in live}
        removed = set()
        for satellite in sorted(groups):
            domain = sorted(groups[satellite] & live)
            for a in domain:
                if a in removed:
                    continue
                for b in domain:
                    if a == b or b in removed:
                        continue
                    wa, wb = float(graph.nodes[a]["weight"]), float(graph.nodes[b]["weight"])
                    if wb >= wa and external[b] <= external[a] and (
                        wb > wa or external[b] < external[a] or b < a
                    ):
                        removed.add(a)
                        records.append({"pass": passes + 1, "removed_slot": a, "witness_slot": b,
                                        "satellite_id": satellite, "removed_weight": wa, "witness_weight": wb})
                        break
        if not removed:
            break
        live.difference_update(removed)
        passes += 1
    return {"graph": graph.subgraph(live), "removals": records, "passes": passes}


phase_started = time.perf_counter()
original_conflict_graph = conflict_graph
global_dominance = reduce_same_satellite_dominance(original_conflict_graph)
# Materialize the reduced root once; recursive reductions remain induced views.
conflict_graph = global_dominance["graph"].copy()
pd.DataFrame(global_dominance["removals"], columns=["pass", "removed_slot", "witness_slot", "satellite_id", "removed_weight", "witness_weight"]).to_csv(RUN_DIR / "global_dominance_removals.csv", index=False)
pd.DataFrame([{"slot_id": v, **d} for v, d in conflict_graph.nodes(data=True)]).to_csv(RUN_DIR / "conflict_graph_nodes.csv", index=False)
pd.DataFrame([{"source": u, "target": v, **d} for u, v, d in conflict_graph.edges(data=True)], columns=["source", "target", "reason", "reconfiguration_time", "satellite_id"]).to_csv(RUN_DIR / "conflict_graph_edges.csv", index=False)
dominance_summary = {
    "input_nodes": original_conflict_graph.number_of_nodes(), "input_edges": original_conflict_graph.number_of_edges(),
    "input_components": nx.number_connected_components(original_conflict_graph),
    "retained_nodes": conflict_graph.number_of_nodes(), "retained_edges": conflict_graph.number_of_edges(),
    "retained_components": nx.number_connected_components(conflict_graph),
    "removed_nodes": len(global_dominance["removals"]), "passes": global_dominance["passes"],
    "original_graph_nodes_file": "original_conflict_graph_nodes.csv",
    "original_graph_edges_file": "original_conflict_graph_edges.csv",
    "solver_graph_nodes_file": "conflict_graph_nodes.csv", "solver_graph_edges_file": "conflict_graph_edges.csv",
}
(RUN_DIR / "dominance_summary.json").write_text(json.dumps(dominance_summary, indent=2), encoding="utf-8")
record_phase_timing("global_dominance_and_export", phase_started, **dominance_summary)
print(dominance_summary)

# ----------------------------------------------------------
# Retrive and display connected components information
# ----------------------------------------------------------

# helper function to print cc summary
def component_summary(component_id: str, graph: nx.Graph) -> dict[str, Any]:
    weights = [data["weight"] for _, data in graph.nodes(data=True)]
    return {
        "component_id": component_id,
        "node_count": graph.number_of_nodes(),
        "edge_count": graph.number_of_edges(),
        "density": nx.density(graph) if graph.number_of_nodes() > 1 else 0.0,
        "total_weight": sum(weights),
        "max_weight": max(weights, default=0.0),
        "satellite_count": len({data["satellite_id"] for _, data in graph.nodes(data=True)}),
        "antenna_count": len({data["antenna_id"] for _, data in graph.nodes(data=True)}),
    }


phase_started = time.perf_counter()
components = [] # where we store connected components

# iterate through the graph and extract CCs
for index, node_ids in enumerate(
    sorted(nx.connected_components(conflict_graph), key=lambda c: (-len(c), sorted(c))),
    start=1,
):
    # print cc summary
    component_id = f"component_{index:03d}"
    graph = conflict_graph.subgraph(sorted(node_ids)).copy()
    components.append({"component_id": component_id, "graph": graph, "summary": component_summary(component_id, graph)})

components_summary = pd.DataFrame([component["summary"] for component in components])
components_summary.to_csv(RUN_DIR / "components_summary.csv", index=False)
components_summary.head(10)
record_phase_timing("natural_components_and_summary", phase_started, component_count=len(components))

[MWIS timing] global_dominance_and_export: 0.72s
{'input_nodes': 5224, 'input_edges': 53998, 'input_components': 4, 'retained_nodes': 4553, 'retained_edges': 43386, 'retained_components': 14, 'removed_nodes': 671, 'passes': 5, 'original_graph_nodes_file': 'original_conflict_graph_nodes.csv', 'original_graph_edges_file': 'original_conflict_graph_edges.csv', 'solver_graph_nodes_file': 'conflict_graph_nodes.csv', 'solver_graph_edges_file': 'conflict_graph_edges.csv'}
[MWIS timing] natural_components_and_summary: 0.20s
Out[0]: 0.2044393999967724


## 5. Exact separator decomposition

Each natural component is independent and can be handled separately. A component that fits both QPU limits becomes a terminal subgraph. Otherwise, the algorithm chooses a small vertex separator and considers every feasible selection state of that separator; removing its conflicts splits the remaining graph into independent pieces that are solved recursively.

Considering every separator state preserves the optimum when decomposition completes. If a configured search budget is reached, the affected node is marked incomplete and later receives a clearly labelled feasible heuristic solution and conservative upper bound.

At every uncached recursive node, the same exact dominance rule is applied again before isolated vertices are fixed. Branch-local reductions are recorded in reconstruction metadata. Separator search and all budgets retain their original behavior.

In [6]:
def materialize_node_graph(node: dict[str, Any]) -> nx.Graph:
    """Recreate a leaf graph as a read-only view of the full conflict graph."""
    return conflict_graph.subgraph(node["graph_node_ids"])


def retain_leaf_graph(node: dict[str, Any], graph: nx.Graph) -> None:
    """Store only the IDs needed to recreate terminal or unresolved graphs."""
    node["graph_node_ids"] = tuple(graph.nodes)


# Profiling helper: count calls and total seconds per decomposition operation
def add_decomposition_time(profile: dict[str, Any], operation: str, started: float) -> None:
    totals = profile.setdefault(operation, {"calls": 0, "seconds": 0.0})
    totals["calls"] += 1
    totals["seconds"] += time.perf_counter() - started


def graph_fits_qpu_limits(graph: nx.Graph) -> bool:
    """
    Return whether both terminal-size limits are satisfied.
    """
    return (
        graph.number_of_nodes() <= MAX_QPU_TERMINAL_NODES
        and graph.number_of_edges() <= MAX_QPU_TERMINAL_COUPLERS
    )


def exact_preprocess(graph: nx.Graph) -> dict[str, Any]:
    """Repeat safe dominance in each branch, then fix isolated vertices."""
    dominance = reduce_same_satellite_dominance(graph)
    surviving = dominance["graph"]
    forced_selected = set(nx.isolates(surviving))
    reduced = surviving.subgraph(set(surviving.nodes) - forced_selected)
    return {"graph": reduced, "forced_selected": forced_selected,
            "forced_objective": sum(graph.nodes[v]["weight"] for v in forced_selected),
            "dominance_removals": dominance["removals"], "dominance_passes": dominance["passes"]}


def residual_components_after_separator(
    graph: nx.Graph, separator: tuple[str, ...], profile: dict[str, Any] | None = None,
) -> list[set[str]]:
    """
    Extract the new sub-problems after removing the graph separator (avoiding copying the whoel graph).
    """    
    started = time.perf_counter()
    # Remove the separator and split what is left into connected pieces
    residual_nodes = set(graph.nodes) - set(separator)
    residual = graph.subgraph(residual_nodes)  # read-only view; connected_components only reads it
    pieces = [set(c) for c in nx.connected_components(residual)]
    if profile is not None:
        add_decomposition_time(profile, "separator_partition_scans", started)
    return pieces


def articulation_piece_sizes(graph: nx.Graph) -> dict[str, tuple[int, ...]]:
    """
    Return the size of the new sub-problems obtained after deleting the articulation node (the node that splits the graph).
    """
    # discovery: order in which the DFS first reaches each node
    # low: earliest discovery reachable from the node's subtree via one back edge
    # parent: node the DFS came from (to backtrack and skip the edge back to it)
    # subtree_size: number of nodes in the node's DFS subtree, itself included
    discovery, low, parent, subtree_size = {}, {}, {}, {}
    
    # Nodes of each connected component, keyed by its DFS root
    members_by_root = {}
    clock = 0

    # Iterative DFS avoids Python recursion limits on long components.
    for root in graph:
        # Start a new DFS only in components not visited yet
        if root in discovery:
            continue
        # Initialise the root of a new component
        clock += 1
        discovery[root] = low[root] = clock
        parent[root] = None
        subtree_size[root] = 1
        members_by_root[root] = [root]
        # Each entry: a node and its neighbours still to explore
        stack = [(root, iter(graph.neighbors(root)))]
        while stack:
            node, neighbors = stack[-1]
            try:
                neighbor = next(neighbors)
            except StopIteration:
                # Node finished: pass its size and low-link up to the parent
                stack.pop()
                ancestor = parent[node]
                if ancestor is not None:
                    subtree_size[ancestor] += subtree_size[node]
                    low[ancestor] = min(low[ancestor], low[node])
                continue

            # Unvisited neighbour: go deeper
            if neighbor not in discovery:
                clock += 1
                discovery[neighbor] = low[neighbor] = clock
                parent[neighbor] = node
                subtree_size[neighbor] = 1
                members_by_root[root].append(neighbor)
                stack.append((neighbor, iter(graph.neighbors(neighbor))))
            elif neighbor != parent[node]:
                # Already visited, not the parent: back edge to an earlier node
                low[node] = min(low[node], discovery[neighbor])

    # Low-link values tell which DFS child subtrees disconnect when a node is removed.
    result = {}
    for root, members in members_by_root.items():
        component_size = subtree_size[root]
        for node in members:
            # Child subtrees with no edge above node become separate pieces
            separated = [
                subtree_size[child]
                for child in graph.neighbors(node)
                if parent.get(child) == node and low[child] >= discovery[node]
            ]
            # Everything else (the ancestors' side) forms one more piece
            if node != root:
                remainder = component_size - 1 - sum(separated)
                if remainder:
                    separated.append(remainder)
            # Two or more pieces: node is an articulation point
            if len(separated) > 1:
                result[node] = tuple(sorted(separated))
    return result


# Subsets of the separator with no internal conflict: one branch each
def feasible_separator_states(graph: nx.Graph, separator: tuple[str, ...]) -> list[frozenset[str]]:
    states = []
    # Try every subset size, from none selected to all selected
    for size in range(len(separator) + 1):
        for selected in combinations(separator, size):
            selected = set(selected)
            # Keep the subset only if its nodes do not conflict with each other
            if all(not graph.has_edge(a, b) for a, b in combinations(selected, 2)):
                states.append(frozenset(selected))
    return states


def separator_piece_sizes(
    adjacency: dict[str, frozenset[str]], separator: tuple[str, ...],
    profile: dict[str, Any] | None = None,
) -> tuple[int, ...]:
    """Compute exact residual component sizes using one cached adjacency table.

    Native set intersections avoid rebuilding and traversing filtered graph views
    for every candidate. Sizes suffice for ranking; the selected separator's
    actual branch graphs are still constructed by the unchanged recursion.
    """
    started = time.perf_counter()
    remaining = set(adjacency).difference(separator)
    sizes = []
    while remaining:
        queue = [remaining.pop()]
        size = 1
        for node in queue:
            reached = adjacency[node].intersection(remaining)
            remaining.difference_update(reached)
            queue.extend(reached)
            size += len(reached)
            if not remaining:
                break  # All remaining vertices have already been discovered.
        sizes.append(size)
    if profile is not None:
        add_decomposition_time(profile, "separator_partition_scans", started)
    return tuple(sizes)


# Rank a separator (lower tuple is better); None if it does not split the graph
def score_separator(
    graph: nx.Graph, separator: tuple[str, ...], pieces: list[set[str]] | None = None,
    profile: dict[str, Any] | None = None,
    adjacency: dict[str, frozenset[str]] | None = None,
) -> tuple | None:
    started = time.perf_counter()
    if pieces is None:
        if adjacency is None:
            adjacency = {node: frozenset(graph.neighbors(node)) for node in graph}
        sizes = separator_piece_sizes(adjacency, separator, profile)
    else:
        sizes = tuple(len(piece) for piece in pieces)
    # Not a real cut: the graph stays in one piece
    if len(sizes) <= 1:
        if profile is not None:
            add_decomposition_time(profile, "separator_scoring", started)
        return None
    # First limit the largest child, then prefer a more balanced split.
    score = (
        max(sizes),
        sum(size * size for size in sizes),
        len(feasible_separator_states(graph, separator)),
        len(separator),
        tuple(sorted(separator)),
    )
    if profile is not None:
        add_decomposition_time(profile, "separator_scoring", started)
    return score


def score_separator_piece_sizes(
    graph: nx.Graph, separator: tuple[str, ...], sizes: tuple[int, ...],
    profile: dict[str, Any] | None = None,
) -> tuple | None:
    """Score an articulation separator without rescanning the graph for its pieces."""
    started = time.perf_counter()
    if len(sizes) <= 1:
        if profile is not None:
            add_decomposition_time(profile, "separator_scoring", started)
        return None
    # Same ranking as score_separator
    score = (
        max(sizes),
        sum(size * size for size in sizes),
        len(feasible_separator_states(graph, separator)),
        len(separator),
        tuple(sorted(separator)),
    )
    if profile is not None:
        add_decomposition_time(profile, "separator_scoring", started)
    return score


# Pick the best separator: articulation points first, then larger cuts
def find_separator(graph: nx.Graph, profile: dict[str, Any] | None = None) -> dict[str, Any] | None:
    candidates = []

    # 1) Single-node cuts, all found in one DFS pass
    articulation_candidates = []
    operation_started = time.perf_counter()
    articulation_sizes = articulation_piece_sizes(graph)
    if profile is not None:
        add_decomposition_time(profile, "articulation_analysis", operation_started)
        profile["articulation_candidates"] = profile.get("articulation_candidates", 0) + len(articulation_sizes)
    for node, sizes in articulation_sizes.items():
        separator = (node,)
        score = score_separator_piece_sizes(graph, separator, sizes, profile)
        if score is not None:
            articulation_candidates.append((score, separator, "articulation point"))

    # Prefer the best-balanced one-node cut before searching larger cuts.
    # This can change the decomposition tree, but not the optimum: every
    # feasible selected/not-selected state of the chosen separator is solved.
    if articulation_candidates:
        score, separator, method = min(articulation_candidates, key=lambda item: item[0])
        return {"separator": separator, "score": score, "method": method}

    # Cache only the current induced graph, once for all candidate cuts.
    operation_started = time.perf_counter()
    adjacency = {node: frozenset(graph.neighbors(node)) for node in graph}
    if profile is not None:
        add_decomposition_time(profile, "separator_adjacency_construction", operation_started)

    # 2) Small graphs: try every pair of nodes
    node_ids = sorted(graph.nodes)
    if MAX_SEPARATOR_SIZE >= 2 and len(node_ids) <= 200:
        for separator in combinations(node_ids, 2):
            if profile is not None:
                profile["pair_candidates"] = profile.get("pair_candidates", 0) + 1
            score = score_separator(graph, separator, profile=profile, adjacency=adjacency)
            if score is not None:
                candidates.append((score, separator, "exhaustive separator pair"))

    # 3) Larger cuts drawn only from the highest-degree nodes
    operation_started = time.perf_counter()
    pool = [
        node for node, _ in sorted(
            graph.degree,
            key=lambda item: (-item[1], item[0]),
        )[:MAX_SEPARATOR_CANDIDATE_POOL]
    ]
    if profile is not None:
        add_decomposition_time(profile, "candidate_pool_construction", operation_started)
    for size in range(2, MAX_SEPARATOR_SIZE + 1):
        for separator in combinations(pool, size):
            if profile is not None:
                profile["degree_pool_candidates"] = profile.get("degree_pool_candidates", 0) + 1
            score = score_separator(graph, separator, profile=profile, adjacency=adjacency)
            if score is not None:
                candidates.append((score, separator, f"degree-pool separator size {size}"))

    # Keep the lowest score; None if nothing splits the graph
    if not candidates:
        return None
    score, separator, method = min(candidates)
    return {"separator": tuple(separator), "score": score, "method": method}


def build_decomposition_node(graph: nx.Graph, component_id: str, context: dict[str, Any], depth: int = 0) -> dict[str, Any]:
    """Build one decomposition-tree node, splitting or recording a terminal/fallback.

    The selected separator's feasible states become branches. Independent pieces
    are solved separately; budget limits are recorded as incomplete leaves.
    """
    # Reuse the result if this exact node set was already decomposed
    cache_key = frozenset(graph.nodes)
    if cache_key in context["cache"]:
        context["cache_hits"] += 1
        return context["cache"][cache_key]

    # Unique ID for this tree node
    node_number = context["next_node_number"]
    context["next_node_number"] += 1
    node_id = f"{component_id}_dnode_{node_number:04d}"

    if node_number % 100 == 0:
        print(f"[MWIS timing] {component_id}: built {node_number:,} decomposition nodes (depth {depth})", flush=True)

    # Take isolated nodes for free and continue on the rest
    operation_started = time.perf_counter()
    preprocessing = exact_preprocess(graph)
    add_decomposition_time(context["profile"], "exact_preprocessing", operation_started)
    reduced = preprocessing["graph"]
    node = {
        "decomposition_node_id": node_id,
        "original_component_id": component_id,
        "depth": depth,
        "input_node_count": graph.number_of_nodes(),
        "input_edge_count": graph.number_of_edges(),
        "graph_node_count": reduced.number_of_nodes(),
        "graph_edge_count": reduced.number_of_edges(),
        "forced_selected": set(preprocessing["forced_selected"]),
        "forced_objective": preprocessing["forced_objective"],
        "dominance_removals": preprocessing["dominance_removals"],
        "dominance_input_nodes": sorted(graph.nodes) if preprocessing["dominance_removals"] else [],
        "children": [],
        "branches": [],
    }
    # Register the node before recursing so repeated subgraphs hit the cache
    context["cache"][cache_key] = node
    context["nodes"][node_id] = node

    # Solved leaves require no further decomposition, even after a budget is exhausted.
    if reduced.number_of_nodes() == 0:
        node.update({"node_type": "terminal", "terminal_reason": "resolved by exact preprocessing"})
        retain_leaf_graph(node, reduced)
        return node

    node_budget_exceeded = context["next_node_number"] > MAX_DECOMPOSITION_NODES
    depth_budget_exceeded = depth > MAX_RECURSION_DEPTH
    # Accept a fitting leaf before a budget rejects further recursion. When budgets
    # remain available, preserve the usual splitting of independent pieces below.
    if (node_budget_exceeded or depth_budget_exceeded) and graph_fits_qpu_limits(reduced):
        node.update({"node_type": "terminal", "terminal_reason": "fits QPU limits"})
        retain_leaf_graph(node, reduced)
        return node

    # Budgets still stop further decomposition of oversized unresolved pieces.
    if node_budget_exceeded:
        node.update({"node_type": "incomplete", "reason": "decomposition node budget exceeded"})
        retain_leaf_graph(node, reduced)
        return node
    if depth_budget_exceeded:
        node.update({"node_type": "incomplete", "reason": "recursion depth exceeded"})
        retain_leaf_graph(node, reduced)
        return node

    # Already disconnected: solve each piece independently and sum
    operation_started = time.perf_counter()
    natural_pieces = [set(c) for c in nx.connected_components(reduced)]
    add_decomposition_time(context["profile"], "natural_component_scans", operation_started)
    if len(natural_pieces) > 1:
        node.update({"node_type": "independent_sum"})
        node["children"] = [
            build_decomposition_node(reduced.subgraph(sorted(piece)), component_id, context, depth + 1)
            for piece in sorted(natural_pieces, key=lambda c: (-len(c), sorted(c)))
        ]
        return node

    # Small enough: this leaf is sent to the QPU
    if graph_fits_qpu_limits(reduced):
        node.update({"node_type": "terminal", "terminal_reason": "fits QPU limits"})
        retain_leaf_graph(node, reduced)
        return node

    # Too big and connected: look for a separator
    operation_started = time.perf_counter()
    separator_result = find_separator(reduced, context["profile"])
    add_decomposition_time(context["profile"], "separator_search", operation_started)
    if separator_result is None:
        node.update({"node_type": "incomplete", "reason": "no separator found within configured search limits"})
        retain_leaf_graph(node, reduced)
        return node

    # One branch per valid selection of separator nodes
    separator = tuple(separator_result["separator"])
    operation_started = time.perf_counter()
    states = feasible_separator_states(reduced, separator)
    add_decomposition_time(context["profile"], "separator_state_generation", operation_started)
    if context["separator_state_count"] + len(states) > MAX_SEPARATOR_STATES:
        node.update({"node_type": "incomplete", "reason": "separator-state budget exceeded"})
        retain_leaf_graph(node, reduced)
        return node

    context["separator_state_count"] += len(states)
    node.update({
        "node_type": "separator",
        "separator": separator,
        "separator_method": separator_result["method"],
        "separator_score": separator_result["score"],
    })

    # For each branch: fix the separator choice and recurse on what is left
    separator_set = set(separator)
    for state_index, selected_separator in enumerate(states):
        selected_separator = set(selected_separator)
        # Neighbours of selected separator nodes cannot be selected
        forced_neighbors = set()
        for selected_node in selected_separator:
            forced_neighbors.update(reduced.neighbors(selected_node))
        forced_neighbors.difference_update(separator_set)

        # Drop the separator and forced-out nodes, then split into independent pieces
        operation_started = time.perf_counter()
        branch_nodes = set(reduced.nodes) - separator_set - forced_neighbors
        branch_graph = reduced.subgraph(branch_nodes)
        branch_components = [set(c) for c in nx.connected_components(branch_graph)]
        add_decomposition_time(context["profile"], "branch_partition_scans", operation_started)
        children = [
            build_decomposition_node(branch_graph.subgraph(sorted(piece)), component_id, context, depth + 1)
            for piece in sorted(branch_components, key=lambda c: (-len(c), sorted(c)))
        ]
        node["branches"].append({
            "state_index": state_index,
            "selected_separator": selected_separator,
            "state_objective": sum(reduced.nodes[n]["weight"] for n in selected_separator),
            "children": children,
        })
    return node


# Decompose one natural component and collect its leaves
def build_component_decomposition(component: dict[str, Any]) -> dict[str, Any]:
    context = {"next_node_number": 1, "cache": {}, "nodes": {}, "cache_hits": 0, "separator_state_count": 0, "profile": {}}
    root = build_decomposition_node(component["graph"], component["component_id"], context)
    nodes = list(context["nodes"].values())
    # Terminal leaves go to the QPU; incomplete ones need a fallback
    terminals = [node for node in nodes if node["node_type"] == "terminal"]
    incomplete = [node for node in nodes if node["node_type"] == "incomplete"]
    return {
        "component_id": component["component_id"],
        "root": root,
        "nodes": nodes,
        "terminals": terminals,
        "incomplete": incomplete,
        "status": "complete" if not incomplete else "incomplete",
        "separator_state_count": context["separator_state_count"],
        "cache_hits": context["cache_hits"],
        "operation_profile": context["profile"],
        "max_depth": max((node["depth"] for node in nodes), default=0),
    }

In [7]:
# Decompose every natural component
phase_started = time.perf_counter()
decompositions = []
for component_index, component in enumerate(components, start=1):
    component_started = time.perf_counter()
    component_id = component["component_id"]
    print(f"[MWIS timing] decomposing {component_id} ({component_index}/{len(components)})", flush=True)
    decomp = build_component_decomposition(component)
    decompositions.append(decomp)
    record_phase_timing("component_decomposition", component_started, component_id=component["component_id"], index=component_index, total=len(components), decomposition_nodes=len(decomp["nodes"]), separator_states=decomp["separator_state_count"], operation_profile=decomp["operation_profile"])
# Collect leaves across all components (empty terminals need no solving)
terminal_nodes = [node for decomp in decompositions for node in decomp["terminals"]]
nonempty_terminal_nodes = [node for node in terminal_nodes if node["graph_node_count"] > 0]
incomplete_nodes = [node for decomp in decompositions for node in decomp["incomplete"]]

# One summary row per component
decomposition_rows = []
for decomp in decompositions:
    root = decomp["root"]
    decomposition_rows.append({
        "component_id": decomp["component_id"],
        "status": decomp["status"],
        "root_type": root["node_type"],
        "input_nodes": root["input_node_count"],
        "input_edges": root["input_edge_count"],
        "terminal_graphs": len([n for n in decomp["terminals"] if n["graph_node_count"] > 0]),
        "incomplete_nodes": len(decomp["incomplete"]),
        "separator_states": decomp["separator_state_count"],
        "cache_hits": decomp["cache_hits"],
        "max_depth": decomp["max_depth"],
    })

pd.DataFrame(decomposition_rows).to_csv(RUN_DIR / "decomposition_summary.csv", index=False)

# Convert a graph to plain JSON: nodes and edges with their attributes
def serialize_graph(graph: nx.Graph) -> dict[str, Any]:
    return {
        "nodes": [{"id": node, **data} for node, data in graph.nodes(data=True)],
        "edges": [{"source": u, "target": v, **data} for u, v, data in graph.edges(data=True)],
    }

# Export the non-empty terminal subgraphs for the annealing notebook
terminal_subgraphs = []
for node in nonempty_terminal_nodes:
    terminal_subgraphs.append({
        "terminal_id": node["decomposition_node_id"],
        "original_component_id": node["original_component_id"],
        "depth": node["depth"],
        "node_count": node["graph_node_count"],
        "edge_count": node["graph_edge_count"],
        "forced_objective": node["forced_objective"],
        "forced_selected": sorted(node["forced_selected"]),
        "terminal_reason": node["terminal_reason"],
        "graph": serialize_graph(materialize_node_graph(node)),
    })

(RUN_DIR / "terminal_subgraphs.json").write_text(json.dumps(terminal_subgraphs, indent=2), encoding="utf-8")

print({
    "natural_components": len(components),
    "complete_decompositions": sum(d["status"] == "complete" for d in decompositions),
    "incomplete_decompositions": sum(d["status"] != "complete" for d in decompositions),
    "terminal_subgraphs": len(nonempty_terminal_nodes),
    "largest_terminal_nodes": max((n["graph_node_count"] for n in nonempty_terminal_nodes), default=0),
    "largest_terminal_couplers": max((n["graph_edge_count"] for n in nonempty_terminal_nodes), default=0),
    "separator_states": sum(d["separator_state_count"] for d in decompositions),
})
record_phase_timing("decomposition_and_terminal_export", phase_started, components=len(decompositions), decomposition_nodes=sum(len(d["nodes"]) for d in decompositions), terminal_subgraphs=len(nonempty_terminal_nodes), separator_states=sum(d["separator_state_count"] for d in decompositions))

[MWIS timing] decomposing component_001 (1/14)
{"utc": "2026-10-12T19:43:42.896793+00:00", "event": "heartbeat", "stage": "mwis", "original_cell": 11, "elapsed_seconds": 20.208, "cpu_seconds": 22.75, "resident_mb": 219.41}
{"utc": "2026-10-12T19:44:03.023301+00:00", "event": "heartbeat", "stage": "mwis", "original_cell": 11, "elapsed_seconds": 40.335, "cpu_seconds": 42.609, "resident_mb": 224.48}
[MWIS timing] component_001: built 100 decomposition nodes (depth 13)
{"utc": "2026-10-12T19:44:23.140069+00:00", "event": "heartbeat", "stage": "mwis", "original_cell": 11, "elapsed_seconds": 60.452, "cpu_seconds": 62.5, "resident_mb": 229.05}
{"utc": "2026-10-12T19:44:43.253001+00:00", "event": "heartbeat", "stage": "mwis", "original_cell": 11, "elapsed_seconds": 80.565, "cpu_seconds": 82.453, "resident_mb": 235.39}
{"utc": "2026-10-12T19:45:03.332025+00:00", "event": "heartbeat", "stage": "mwis", "original_cell": 11, "elapsed_seconds": 100.645, "cpu_seconds": 102.297, "resident_mb": 238.98}

## 6. QUBO translation of the terminals

Every non-empty terminal subgraph is translated into a QUBO with one binary variable per contact slot (`x = 1` means the slot is scheduled):

- **reward**: each slot gets the linear term `-qubo_reward_factor * weight`, so selecting heavier slots lowers the energy;
- **conflict penalty**: each conflict edge gets the quadratic term `P = qubo_penalty_factor * (max effective reward + 1)`, computed per terminal. Since `P` exceeds any single reward, dropping one endpoint of a violated edge always lowers the energy, so the QUBO minimum is an independent set and a maximum-weight one.

Each QUBO is saved in `mwis_terminal_qubos.json` in the same layout as the CSP pipeline: the coefficients as `[i, j, coefficient]` rows over integer bits (diagonal rows are the linear terms), and an `index_map` `{slot_id: [bit, next_bit]}`. A slot is chosen in a sample when `sample[bit] - sample[next_bit]` equals 1; MWIS uses one bit per slot, so `next_bit` is always null. Each record also carries `slot_weights` and `conflicts` (the slot pairs that cannot be selected together), so a decoded sample can be scored and checked without the graph. Together with the terminal's forced selections, this is everything needed to turn an annealing sample back into slots of the original problem.

In [8]:
phase_started = time.perf_counter()

def max_effective_reward(graph: nx.Graph) -> float:
    """
    Largest scaled node reward in the terminal graph given.
    """
    rewards = [REWARD_FACTOR * float(data["weight"]) for _, data in graph.nodes(data=True)]
    return max([0.0, *rewards])


def default_penalty(graph: nx.Graph) -> float:
    """
    Compute the conflict penalty, relative to the largest effective reward.
    """
    return PENALTY_FACTOR * (max_effective_reward(graph) + 1.0)


def build_qubo_from_graph(graph: nx.Graph, penalty: float | None = None) -> tuple[dict[tuple[int, int], float], dict[str, tuple[int, None]]]:
    """
    Build the QUBO of a terminal graph: one bit per slot, a reward on the diagonal and a
    penalty on every conflict edge. Returns the QUBO dict {(i, j): coefficient} and the
    index map {slot_id: (bit, None)}.
    """
    if penalty is None:
        penalty = default_penalty(graph)

    # one bit per slot, in a fixed order
    bit = {slot_id: index for index, slot_id in enumerate(sorted(graph.nodes))}

    # linear reward
    Q = {(bit[node], bit[node]): -REWARD_FACTOR * float(data["weight"]) for node, data in graph.nodes(data=True)}

    # quadratic penalty
    for u, v in graph.edges:
        i, j = sorted((bit[u], bit[v]))
        Q[(i, j)] = float(penalty)

    return Q, {slot_id: (index, None) for slot_id, index in bit.items()}


terminal_qubos = []
for node in nonempty_terminal_nodes:
    graph = materialize_node_graph(node)
    Q, index_map = build_qubo_from_graph(graph)
    terminal_qubos.append({
        "terminal_id": node["decomposition_node_id"],
        "original_component_id": node["original_component_id"],
        "encoding": "one_hot",                                   # one bit per slot
        "qubits": len(index_map),
        "couplers": sum(1 for i, j in Q if i != j),
        "max_effective_reward": max_effective_reward(graph),
        "penalty": default_penalty(graph),
        "forced_selected": sorted(node["forced_selected"]),      # exact selections, added to the decoded sample
        "forced_objective": node["forced_objective"],
        "qubo": [[i, j, coefficient] for (i, j), coefficient in sorted(Q.items())],   # {(i, j): coeff} as rows
        "index_map": {slot_id: list(bits) for slot_id, bits in index_map.items()},  # slot_id -> [bit, next_bit]
        # to score a decoded sample: slot weights and the slot pairs that cannot be selected together
        "slot_weights": {slot_id: float(data["weight"]) for slot_id, data in graph.nodes(data=True)},
        "conflicts": sorted(sorted((u, v)) for u, v in graph.edges),
    })

(RUN_DIR / "mwis_terminal_qubos.json").write_text(json.dumps(terminal_qubos), encoding="utf-8")
record_phase_timing("qubo_translation", phase_started, terminals=len(terminal_qubos))

qubo_sizes = pd.DataFrame([
    {
        **{k: t[k] for k in ("terminal_id", "original_component_id", "qubits", "couplers", "max_effective_reward", "penalty")},
        "penalty_margin": t["penalty"] - t["max_effective_reward"],
    }
    for t in terminal_qubos
])
print(f"Terminal QUBOs: {len(terminal_qubos)}")
if not qubo_sizes.empty:
    print(f"  qubits   max {qubo_sizes['qubits'].max()} (limit {MAX_QPU_TERMINAL_NODES})")
    print(f"  couplers max {qubo_sizes['couplers'].max()} (limit {MAX_QPU_TERMINAL_COUPLERS})")
qubo_sizes.head(10)

[MWIS timing] qubo_translation: 0.00s
Terminal QUBOs: 6
  qubits   max 7 (limit 30)
  couplers max 13 (limit 200)
Out[0]: 
                 terminal_id original_component_id  ...  penalty  penalty_margin
0   component_001_dnode_0527         component_001  ...   84.611             1.0
1   component_001_dnode_1060         component_001  ...   60.709             1.0
2   component_001_dnode_4121         component_001  ...   92.172             1.0
3   component_001_dnode_8217         component_001  ...   71.482             1.0
4  component_001_dnode_10007         component_001  ...   61.917             1.0
5   component_002_dnode_0001         component_002  ...   85.813             1.0

[6 rows x 7 columns]


## 7. MWIS solvers and reconstruction

The tree is evaluated from its leaves upward. Small terminal graphs use exact classical solvers within the configured guards; larger terminal graphs use the deterministic greedy solver for this classical baseline. Incomplete decomposition leaves also use greedy, but remain explicitly non-exact and carry a conservative upper bound. Separator alternatives are compared, and independent pieces are summed. A per-natural-component objective and upper bound are saved for the later annealing/MILP hybrid report.

This phase also saves a compact reconstruction map so the annealing notebook can later combine terminal QUBO samples back into one MWIS schedule.

In [9]:
# Total weight of a set of slots
def node_set_weight(graph: nx.Graph, selected: set[str]) -> float:
    return sum(graph.nodes[node]["weight"] for node in selected)


# True if no two selected slots conflict
def is_independent(graph: nx.Graph, selected: set[str]) -> bool:
    return all(not graph.has_edge(u, v) for u, v in combinations(selected, 2))


# Runtime profiling: log file and counters for solver calls and tree evaluation
PROFILE_PATH = RUN_DIR / f'mwis_runtime_profile_{time.strftime("%Y%m%d_%H%M%S")}.jsonl'
SOLVER_PROFILE = {"calls": Counter(), "seconds": Counter(), "max_seconds": Counter()}
EVALUATION_PROFILE = {"calls": 0, "completed": 0, "memo_hits": 0, "calls_by_type": Counter(), "inclusive_seconds_by_type": Counter()}
# Fallback solutions of incomplete leaves, reused when exporting metadata
INCOMPLETE_LEAF_METADATA: dict[str, dict[str, Any]] = {}

# Append one profiling event to the JSONL log
def log_profile_event(event: str, **details: Any) -> None:
    with PROFILE_PATH.open("a", encoding="utf-8") as profile_file:
        profile_file.write(json.dumps({"event": event, **details}) + "\n")
        profile_file.flush()


# Exact: try every subset and keep the heaviest independent one (tiny graphs only)
def solve_mwis_enumeration(graph: nx.Graph, terminal_id: str) -> dict[str, Any]:
    nodes = sorted(graph.nodes)
    best_nodes = set()
    best_value = 0.0
    checked = 0
    total = 1 << len(nodes)
    started = time.perf_counter()
    for size in range(len(nodes) + 1):
        for subset in combinations(nodes, size):
            checked += 1
            # Periodic progress log for long enumerations
            if checked % 500_000 == 0:
                elapsed = time.perf_counter() - started
                log_profile_event("enumeration_progress", terminal_id=terminal_id, nodes=len(nodes), checked=checked, total=total, elapsed_seconds=round(elapsed, 3))
                print(f"[MWIS profile] {terminal_id}: checked {checked:,}/{total:,} subsets in {elapsed:.1f}s", flush=True)
            selected = set(subset)
            if is_independent(graph, selected):
                value = node_set_weight(graph, selected)
                if value > best_value:
                    best_nodes, best_value = selected, value
    return {"status": "optimal", "selected_nodes": best_nodes, "objective": best_value, "solver": "exact enumeration"}


# Exact: an MWIS is a maximum-weight clique in the complement graph
def solve_mwis_clique(graph: nx.Graph) -> dict[str, Any]:
    complement = nx.complement(graph)
    for node, data in graph.nodes(data=True):
        # The clique solver needs integer weights
        complement.nodes[node]["scaled_weight"] = int(round(data["weight"] * 1000))
    selected, _ = nx.algorithms.clique.max_weight_clique(complement, weight="scaled_weight")
    selected = set(selected)
    return {
        "status": "optimal",
        "selected_nodes": selected,
        "objective": node_set_weight(graph, selected),
        "solver": "complement max-weight clique",
    }


# Heuristic: pick nodes by weight per conflict, skipping neighbours of chosen ones
def solve_mwis_greedy(graph: nx.Graph) -> dict[str, Any]:
    # Priority is static, so sorting once preserves the old repeated-max order.
    priority_order = sorted(
        graph.nodes,
        key=lambda node: (
            graph.nodes[node]["weight"] / (1 + graph.degree(node)),
            graph.nodes[node]["weight"],
            -graph.degree(node),
            node,
        ),
        reverse=True,
    )
    unavailable = set()
    selected = set()
    for node in priority_order:
        if node in unavailable:
            continue
        selected.add(node)
        unavailable.add(node)
        unavailable.update(graph.neighbors(node))
    return {
        "status": "heuristic_feasible",
        "selected_nodes": selected,
        "objective": node_set_weight(graph, selected),
        "solver": "deterministic weighted greedy",
    }


# Choose the solver by graph size: enumeration, then clique, then greedy
def solve_terminal_mwis(graph: nx.Graph, terminal_id: str) -> dict[str, Any]:
    node_count = graph.number_of_nodes()
    if node_count == 0:
        solver_name = "empty graph"
        solve = lambda: {"status": "optimal", "selected_nodes": set(), "objective": 0.0, "solver": solver_name}
    elif node_count <= EXACT_ENUMERATION_NODE_LIMIT:
        solver_name = "exact enumeration"
        solve = lambda: solve_mwis_enumeration(graph, terminal_id)
    elif node_count <= EXACT_CLIQUE_NODE_LIMIT:
        solver_name = "complement max-weight clique"
        solve = lambda: solve_mwis_clique(graph)
    else:
        solver_name = "deterministic weighted greedy"
        solve = lambda: solve_mwis_greedy(graph)

    # Run the solver and record its time
    started = time.perf_counter()
    log_profile_event("terminal_solver_start", terminal_id=terminal_id, solver=solver_name, nodes=node_count, edges=graph.number_of_edges())
    print(f"[MWIS profile] start {terminal_id}: {solver_name}, {node_count} nodes, {graph.number_of_edges()} edges", flush=True)
    try:
        result = solve()
    finally:
        elapsed = time.perf_counter() - started
        SOLVER_PROFILE["calls"][solver_name] += 1
        SOLVER_PROFILE["seconds"][solver_name] += elapsed
        SOLVER_PROFILE["max_seconds"][solver_name] = max(SOLVER_PROFILE["max_seconds"][solver_name], elapsed)
        log_profile_event("terminal_solver_finish", terminal_id=terminal_id, solver=solver_name, nodes=node_count, edges=graph.number_of_edges(), elapsed_seconds=round(elapsed, 3))
        print(f"[MWIS profile] done {terminal_id}: {solver_name} in {elapsed:.2f}s", flush=True)
    return result


# Upper bound: total weight of all nodes (ignores conflicts)
def conservative_upper_bound(graph: nx.Graph) -> float:
    return sum(data["weight"] for _, data in graph.nodes(data=True))

In [10]:
# Solve one decomposition-tree node bottom-up
def _evaluate_node(node: dict[str, Any], memo: dict[str, dict[str, Any]] | None = None) -> dict[str, Any]:
    if memo is None:
        memo = {}
    node_id = node["decomposition_node_id"]
    # Shared subtrees are evaluated only once
    if node_id in memo:
        return memo[node_id]

    # Isolated nodes selected during preprocessing
    base_selected = set(node.get("forced_selected", set()))
    base_objective = float(node.get("forced_objective", 0.0))

    # Incomplete leaf: greedy fallback with a conservative upper bound
    if node["node_type"] == "incomplete":
        node_graph = materialize_node_graph(node)
        result = solve_mwis_greedy(node_graph)
        upper_bound = base_objective + conservative_upper_bound(node_graph)
        INCOMPLETE_LEAF_METADATA[node_id] = {
            "status": "heuristic_feasible",
            "forced_selected": sorted(base_selected),
            "heuristic_selected_nodes": sorted(result["selected_nodes"]),
            "selected_nodes": sorted(base_selected | result["selected_nodes"]),
            "objective": base_objective + result["objective"],
            "upper_bound": upper_bound,
        }
        output = {
            "status": "heuristic_feasible",
            "selected_nodes": base_selected | result["selected_nodes"],
            "objective": base_objective + result["objective"],
            "upper_bound": upper_bound,
            "exact_objective": 0.0,
            "heuristic_objective": result["objective"],
            "terminal_ids": set(),
            "heuristic_ids": {node_id},
            "fallback_details": [{
                "node_id": node_id,
                "component_id": node["original_component_id"],
                "reason": node["reason"],
                "node_count": node["graph_node_count"],
                "edge_count": node["graph_edge_count"],
                "forced_selected": sorted(base_selected),
                "heuristic_selected_nodes": sorted(result["selected_nodes"]),
                "selected_nodes": sorted(base_selected | result["selected_nodes"]),
                "heuristic_objective": result["objective"],
                "upper_bound": upper_bound,
            }],
        }
        memo[node_id] = output
        return output

    # Terminal leaf: solve it with the classical baseline solvers
    if node["node_type"] == "terminal":
        node_graph = materialize_node_graph(node)
        result = solve_terminal_mwis(node_graph, node_id)
        is_exact = result["status"] == "optimal"
        objective = base_objective + result["objective"]
        output = {
            "status": "optimal" if is_exact else "heuristic_feasible",
            "selected_nodes": base_selected | result["selected_nodes"],
            "objective": objective,
            "upper_bound": objective if is_exact else base_objective + conservative_upper_bound(node_graph),
            "exact_objective": result["objective"] if is_exact else 0.0,
            "heuristic_objective": 0.0 if is_exact else result["objective"],
            "terminal_ids": {node_id} if node["graph_node_count"] and is_exact else set(),
            "heuristic_ids": set(),
            "fallback_details": [],
        }
        memo[node_id] = output
        return output

    # Independent pieces: combine by summing
    if node["node_type"] == "independent_sum":
        children = [evaluate_node(child, memo) for child in node["children"]]
        output = {
            "status": "optimal" if all(child["status"] == "optimal" for child in children) else "heuristic_feasible",
            "selected_nodes": base_selected | set().union(*(child["selected_nodes"] for child in children)),
            "objective": base_objective + sum(child["objective"] for child in children),
            "upper_bound": base_objective + sum(child["upper_bound"] for child in children),
            "exact_objective": sum(child["exact_objective"] for child in children),
            "heuristic_objective": sum(child["heuristic_objective"] for child in children),
            "terminal_ids": set().union(*(child["terminal_ids"] for child in children)),
            "heuristic_ids": set().union(*(child["heuristic_ids"] for child in children)),
            "fallback_details": [detail for child in children for detail in child["fallback_details"]],
        }
        memo[node_id] = output
        return output

    # Separator node: evaluate every branch
    branch_results = []
    branch_upper_bounds = []
    for branch in node["branches"]:
        children = [evaluate_node(child, memo) for child in branch["children"]]
        # Each branch uses its own separator choice and its own children only
        selected = base_selected | set(branch["selected_separator"])
        for child in children:
            selected.update(child["selected_nodes"])

        objective = base_objective + branch["state_objective"] + sum(child["objective"] for child in children)
        upper_bound = base_objective + branch["state_objective"] + sum(child["upper_bound"] for child in children)
        branch_upper_bounds.append(upper_bound)
        branch_results.append({
            "status": "optimal" if all(child["status"] == "optimal" for child in children) else "heuristic_feasible",
            "selected_nodes": selected,
            "objective": objective,
            "upper_bound": upper_bound,
            "exact_objective": branch["state_objective"] + sum(child["exact_objective"] for child in children),
            "heuristic_objective": sum(child["heuristic_objective"] for child in children),
            "terminal_ids": set().union(*(child["terminal_ids"] for child in children)) if children else set(),
            "heuristic_ids": set().union(*(child["heuristic_ids"] for child in children)) if children else set(),
            "fallback_details": [detail for child in children for detail in child["fallback_details"]],
        })

    # Keep the best branch; the upper bound must cover every branch
    best = max(branch_results, key=lambda item: (item["objective"], sorted(item["selected_nodes"])))
    best["upper_bound"] = max(branch_upper_bounds, default=best["objective"])
    # Not certified optimal if another branch could still be better
    if best["upper_bound"] > best["objective"] + 1e-9:
        best["status"] = "heuristic_feasible"
    memo[node_id] = best
    return best


# Profiling wrapper around _evaluate_node
def evaluate_node(node: dict[str, Any], memo: dict[str, dict[str, Any]] | None = None) -> dict[str, Any]:
    if memo is None:
        memo = {}
    node_id = node["decomposition_node_id"]
    if node_id in memo:
        EVALUATION_PROFILE["memo_hits"] += 1
        return memo[node_id]

    EVALUATION_PROFILE["calls"] += 1
    node_type = node["node_type"]
    EVALUATION_PROFILE["calls_by_type"][node_type] += 1
    started = time.perf_counter()
    result = _evaluate_node(node, memo)
    elapsed = time.perf_counter() - started
    EVALUATION_PROFILE["completed"] += 1
    EVALUATION_PROFILE["inclusive_seconds_by_type"][node_type] += elapsed
    if EVALUATION_PROFILE["completed"] % 25 == 0:
        progress = {"completed": EVALUATION_PROFILE["completed"], "calls": EVALUATION_PROFILE["calls"], "memo_hits": EVALUATION_PROFILE["memo_hits"], "node_type": node_type, "elapsed_seconds": round(elapsed, 3)}
        log_profile_event("evaluation_progress", **progress)
        print(f"[MWIS profile] evaluated {progress['completed']:,} nodes; latest {node_type} took {elapsed:.2f}s", flush=True)
    return result


# Evaluate every component from its root
evaluation_started = time.perf_counter()
log_profile_event("evaluation_start", decomposition_nodes=sum(len(d["nodes"]) for d in decompositions), terminal_subgraphs=len(nonempty_terminal_nodes))
component_results = [evaluate_node(decomp["root"]) for decomp in decompositions]
evaluation_wall_seconds = time.perf_counter() - evaluation_started
log_profile_event("evaluation_finish", wall_seconds=round(evaluation_wall_seconds, 3), **{"calls": EVALUATION_PROFILE["calls"], "completed": EVALUATION_PROFILE["completed"], "memo_hits": EVALUATION_PROFILE["memo_hits"]})
record_phase_timing("recursive_solution_evaluation", evaluation_started, evaluated_nodes=EVALUATION_PROFILE["completed"], memo_hits=EVALUATION_PROFILE["memo_hits"])
# Merge the component results into one global schedule
selected_nodes = set().union(*(result["selected_nodes"] for result in component_results))
final_objective = sum(result["objective"] for result in component_results)
final_upper_bound = sum(result["upper_bound"] for result in component_results)

solution = {
    "status": "optimal" if all(result["status"] == "optimal" for result in component_results) else "heuristic_feasible",
    "selected_nodes": selected_nodes,
    "objective": final_objective,
    "upper_bound": final_upper_bound,
    "absolute_gap_bound": final_upper_bound - final_objective,
    "relative_gap_bound_percent": 0.0 if final_upper_bound == 0 else 100 * (final_upper_bound - final_objective) / final_upper_bound,
    "exact_component_count": sum(result["status"] == "optimal" for result in component_results),
    "heuristic_component_count": sum(result["status"] != "optimal" for result in component_results),
    "exact_components_objective_sum": sum(result["objective"] for result in component_results if result["status"] == "optimal"),
    "heuristic_components_objective_sum": sum(result["objective"] for result in component_results if result["status"] != "optimal"),
    "heuristic_components_upper_bound": sum(result["upper_bound"] for result in component_results if result["status"] != "optimal"),
    "terminal_ids": set().union(*(result["terminal_ids"] for result in component_results)),
    "heuristic_ids": set().union(*(result["heuristic_ids"] for result in component_results)),
    "fallback_details": [detail for result in component_results for detail in result["fallback_details"]],
}

# Sanity check: the final schedule must contain no conflicts
if not is_independent(original_conflict_graph, solution["selected_nodes"]):
    raise AssertionError("Reconstructed solution is not an independent set.")

# Export the selected slots and the per-component results
export_started = time.perf_counter()
selected_slots = slots[slots["slot_id"].isin(solution["selected_nodes"])].copy()
selected_slots.to_csv(RUN_DIR / "mwis_solution.csv", index=False)
component_mwis_summary = pd.DataFrame([
    {
        "component_id": decomp["component_id"],
        "status": result["status"],
        "objective": float(result["objective"]),
        "upper_bound": float(result["upper_bound"]),
        "exact": result["status"] == "optimal",
    }
    for decomp, result in zip(decompositions, component_results)
])
component_mwis_summary.to_csv(RUN_DIR / "component_mwis_summary.csv", index=False)
pd.DataFrame(solution["fallback_details"]).to_csv(RUN_DIR / "heuristic_fallback_details.csv", index=False)


def heuristic_leaf_metadata(node: dict[str, Any]) -> dict[str, Any]:
    """Reuse the fallback computed during evaluation instead of solving it again."""
    return INCOMPLETE_LEAF_METADATA[node["decomposition_node_id"]]


# Compact record of a tree node, used by the annealing notebook's reconstruction
def reconstruction_node_metadata(node: dict[str, Any]) -> dict[str, Any]:
    record = {
        "node_id": node["decomposition_node_id"],
        "component_id": node["original_component_id"],
        "node_type": node["node_type"],
        "depth": node["depth"],
        "input_node_count": node["input_node_count"],
        "input_edge_count": node["input_edge_count"],
        "qpu_node_count": node["graph_node_count"],
        "qpu_edge_count": node["graph_edge_count"],
        "forced_selected": sorted(node.get("forced_selected", set())),
        "forced_objective": float(node.get("forced_objective", 0.0)),
        "dominance_removals": node.get("dominance_removals", []),
        "dominance_input_nodes": node.get("dominance_input_nodes", []),
    }
    if node["node_type"] == "terminal":
        record["terminal_reason"] = node["terminal_reason"]
        record["terminal_id"] = node["decomposition_node_id"] if node["graph_node_count"] else None
    elif node["node_type"] == "incomplete":
        record["reason"] = node["reason"]
        record["heuristic_solution"] = heuristic_leaf_metadata(node)
    elif node["node_type"] == "independent_sum":
        record["children"] = [child["decomposition_node_id"] for child in node["children"]]
    elif node["node_type"] == "separator":
        record["separator"] = list(node["separator"])
        record["separator_method"] = node["separator_method"]
        record["branches"] = [
            {
                "state_index": branch["state_index"],
                "selected_separator": sorted(branch["selected_separator"]),
                "state_objective": float(branch["state_objective"]),
                "children": [child["decomposition_node_id"] for child in branch["children"]],
            }
            for branch in node["branches"]
        ]
    return record


# Reconstruction map: component roots plus every tree node
reconstruction_metadata = {
    "schema_version": 1,
    "variant": "same-satellite dominance before components and during recursion",
    "dominance_summary": dominance_summary,
    "producer": "mwis_pipeline",
    "run_name": RUN_NAME,
    "objective": "maximum-weight independent set",
    "terminal_subgraphs_file": "terminal_subgraphs.json",
    "qubo_encoding": "one_hot",
    "terminal_qubos_file": "mwis_terminal_qubos.json",
    "components": [
        {
            "component_id": decomp["component_id"],
            "root_node_id": decomp["root"]["decomposition_node_id"],
            "status": decomp["status"],
        }
        for decomp in decompositions
    ],
    "nodes": [
        reconstruction_node_metadata(node)
        for decomp in decompositions
        for node in decomp["nodes"]
    ],
}
# Gzip is lossless and keeps large decomposition metadata small enough to share.
with gzip.open(RUN_DIR / "reconstruction_metadata.json.gz", "wt", encoding="utf-8", compresslevel=6) as metadata_file:
    json.dump(reconstruction_metadata, metadata_file)

# Runtime profile summary
profile_summary = {
    "run_name": RUN_NAME,
    "evaluation_wall_seconds": round(evaluation_wall_seconds, 3),
    "evaluation_calls": EVALUATION_PROFILE["calls"],
    "evaluation_completed": EVALUATION_PROFILE["completed"],
    "evaluation_memo_hits": EVALUATION_PROFILE["memo_hits"],
    "evaluation_calls_by_type": dict(EVALUATION_PROFILE["calls_by_type"]),
    "evaluation_inclusive_seconds_by_type": {key: round(value, 3) for key, value in EVALUATION_PROFILE["inclusive_seconds_by_type"].items()},
    "terminal_solver_calls": dict(SOLVER_PROFILE["calls"]),
    "terminal_solver_seconds": {key: round(value, 3) for key, value in SOLVER_PROFILE["seconds"].items()},
    "terminal_solver_max_seconds": {key: round(value, 3) for key, value in SOLVER_PROFILE["max_seconds"].items()},
    "trace_file": PROFILE_PATH.name,
}
(RUN_DIR / "mwis_runtime_profile_summary.json").write_text(json.dumps(profile_summary, indent=2), encoding="utf-8")
record_phase_timing("solution_reconstruction_and_exports", export_started, selected_contacts=len(solution["selected_nodes"]))
print(profile_summary)

print({
    "status": solution["status"],
    "selected_contacts": len(solution["selected_nodes"]),
    "objective": solution["objective"],
    "upper_bound": solution["upper_bound"],
    "relative_gap_bound_percent": solution["relative_gap_bound_percent"],
    "heuristic_subgraphs": len(solution["heuristic_ids"]),
})

[MWIS profile] start component_001_dnode_0015: empty graph, 0 nodes, 0 edges
[MWIS profile] done component_001_dnode_0015: empty graph in 0.00s
[MWIS profile] start component_001_dnode_0017: empty graph, 0 nodes, 0 edges
[MWIS profile] done component_001_dnode_0017: empty graph in 0.00s
[MWIS profile] start component_001_dnode_0020: empty graph, 0 nodes, 0 edges
[MWIS profile] done component_001_dnode_0020: empty graph in 0.00s
[MWIS profile] start component_001_dnode_0022: empty graph, 0 nodes, 0 edges
[MWIS profile] done component_001_dnode_0022: empty graph in 0.00s
[MWIS profile] start component_001_dnode_0023: empty graph, 0 nodes, 0 edges
[MWIS profile] done component_001_dnode_0023: empty graph in 0.00s
[MWIS profile] start component_001_dnode_0031: empty graph, 0 nodes, 0 edges
[MWIS profile] done component_001_dnode_0031: empty graph in 0.00s
[MWIS profile] evaluated 25 nodes; latest incomplete took 0.09s
[MWIS profile] start component_001_dnode_0047: empty graph, 0 nodes, 0 e

## 8. Final summary table

This table is the main compact output of the notebook. The graphics and QUBO notebooks read the artifacts saved above instead of rebuilding this pipeline.

In [11]:
# Components that already fit the QPU (their root is a terminal)
direct_component_ids = {
    decomp["component_id"]
    for decomp in decompositions
    if decomp["root"]["node_type"] == "terminal"
}
# Split terminals into direct ones and ones produced by separator decomposition
directly_solved_terminals = [
    node for node in nonempty_terminal_nodes
    if node["original_component_id"] in direct_component_ids
]
separator_terminals = [
    node for node in nonempty_terminal_nodes
    if node["original_component_id"] not in direct_component_ids
]

# How much of the decomposition hit a budget limit
incomplete_decomposition_nodes = sum(
    int(node.get("node_type") == "incomplete")
    for decomp in decompositions
    for node in decomp["nodes"]
)
components_with_incomplete_decomposition = sum(
    decomp["status"] != "complete" for decomp in decompositions
)

# Final metrics as (name, value, meaning)
final_summary_rows = [
    ("Original candidate nodes", original_conflict_graph.number_of_nodes(), "All input candidate contacts."),
    ("Original conflict edges", original_conflict_graph.number_of_edges(), "All conflicts before dominance."),
    ("Contacts removed by global dominance", dominance_summary["removed_nodes"], "Exact removal of dominated alternatives."),
    ("Contacts retained after global dominance", conflict_graph.number_of_nodes(), "Solver graph nodes before component preprocessing."),
    ("Number of components (total)", len(components), "Connected components after exact dominance."),
    ("Number of couplers (total)", conflict_graph.number_of_edges(), "Conflict edges in the reduced solver graph."),
    ("Total number of solution subgraphs", len(directly_solved_terminals) + len(separator_terminals) + len(solution["heuristic_ids"]), "Terminal subgraphs plus fallback nodes used in the selected solution."),
    ("Subgraphs solved directly", len(directly_solved_terminals), "Natural components that already fit the QPU limits."),
    ("Subgraphs solved exactly with separator decomposition", len(separator_terminals), "Terminal subgraphs produced from oversized components."),
    ("Subgraphs solved heuristically", len(solution["heuristic_ids"]), "Fallback nodes used in the selected feasible solution; unresolved alternative states are counted separately below."),
    ("Components with incomplete separator decomposition", components_with_incomplete_decomposition, "Components with at least one unresolved separator state, including alternatives not selected in the feasible solution."),
    ("Incomplete decomposition nodes (all states)", incomplete_decomposition_nodes, "Unresolved decomposition nodes across all separator states."),
    ("Exact components objective sum", solution["exact_components_objective_sum"], "Objective contribution from components whose decomposition and terminal solves are certified exact."),
    ("Exact global objective", solution["objective"] if solution["status"] == "optimal" else "not certified", "Available only when the global optimum is certified."),
    ("Incomplete/heuristic components feasible sum", solution["heuristic_components_objective_sum"], "Feasible objective from uncertified components; includes their full selected solution, not only fallback-node weight."),
    ("Incomplete/heuristic components conservative upper bound", solution["heuristic_components_upper_bound"], "Conservative upper bound for uncertified components, including unresolved alternative separator states."),
    ("Total feasible objective / lower bound", solution["objective"], "Objective of the reconstructed feasible schedule."),
    ("Total conservative upper bound", solution["upper_bound"], "Conservative upper bound on the true optimum."),
    ("Absolute optimality-gap bound", solution["absolute_gap_bound"], "Upper bound minus feasible objective."),
    ("Relative optimality-gap bound (%)", solution["relative_gap_bound_percent"], "Gap as a percentage of the conservative upper bound."),
]

final_summary = pd.DataFrame(final_summary_rows, columns=["metric", "value", "meaning"])
phase_started = time.perf_counter()
final_summary.to_csv(RUN_DIR / "final_summary.csv", index=False)
record_phase_timing("final_reporting_export", phase_started, metrics=len(final_summary))
record_phase_timing("pipeline_total_so_far", PIPELINE_STARTED, run_name=RUN_NAME)

# Report the shared annealing target without changing it (it may intentionally target CSP).
print(f"MWIS experiment saved: {RUN_NAME}")
annealing_target = run_config.get("qubo_input_run")
if annealing_target and (RESULTS_DIR / annealing_target).resolve() == RUN_DIR.resolve():
    print("The shared annealing input matches this MWIS experiment.")
else:
    print(f"The shared annealing notebook currently targets: {annealing_target or '(not configured)'}")
    print("To anneal this MWIS experiment, set qubo_input_run to:")
    print(RUN_DIR.relative_to(RESULTS_DIR).as_posix())

final_summary

[MWIS timing] final_reporting_export: 0.00s
[MWIS timing] pipeline_total_so_far: 7804.98s
MWIS experiment saved: exp2_s500_g50
The shared annealing input matches this MWIS experiment.
Out[0]: 
                                               metric  ...                                            meaning
0                            Original candidate nodes  ...                      All input candidate contacts.
1                             Original conflict edges  ...                    All conflicts before dominance.
2                Contacts removed by global dominance  ...           Exact removal of dominated alternatives.
3            Contacts retained after global dominance  ...  Solver graph nodes before component preprocess...
4                        Number of components (total)  ...        Connected components after exact dominance.
5                          Number of couplers (total)  ...        Conflict edges in the reduced solver graph.
6                  Total number of so